
# 185. Department Top Three Salaries

**Difficulty:** Hard

---

## 🧩 Problem Description

### Table: `Employee`

| Column Name  | Type    |
| ------------ | ------- |
| id           | int     |
| name         | varchar |
| salary       | int     |
| departmentId | int     |

* `id` is the primary key.
* `departmentId` is a foreign key referencing `Department.id`.
* Each row represents an employee, their salary, and their department.

---

### Table: `Department`

| Column Name | Type    |
| ----------- | ------- |
| id          | int     |
| name        | varchar |

* `id` is the primary key.
* Each row represents a department.

---

## 🎯 Objective

A **high earner** is an employee whose salary is among the **top 3 unique salaries within their department**.

Write an SQL query to return all employees who are high earners in their respective departments.

* Return the result in **any order**.
* The output format should match the example below.

---

## 📥 Example

### Input

**Employee**

| id | name  | salary | departmentId |
| -- | ----- | ------ | ------------ |
| 1  | Joe   | 85000  | 1            |
| 2  | Henry | 80000  | 2            |
| 3  | Sam   | 60000  | 2            |
| 4  | Max   | 90000  | 1            |
| 5  | Janet | 69000  | 1            |
| 6  | Randy | 85000  | 1            |
| 7  | Will  | 70000  | 1            |

**Department**

| id | name  |
| -- | ----- |
| 1  | IT    |
| 2  | Sales |

---

### Output

| Department | Employee | Salary |
| ---------- | -------- | ------ |
| IT         | Max      | 90000  |
| IT         | Joe      | 85000  |
| IT         | Randy    | 85000  |
| IT         | Will     | 70000  |
| Sales      | Henry    | 80000  |
| Sales      | Sam      | 60000  |

---

### Explanation

#### IT Department

* Max earns the highest unique salary.
* Randy and Joe share the second-highest unique salary.
* Will earns the third-highest unique salary.

#### Sales Department

* Henry earns the highest salary.
* Sam earns the second-highest salary.
* There is no third-highest salary since only two employees exist.

---

## ⚠️ Constraints

* No two employees have the same combination of **name, salary, and department**.

---

If you want, I can also format this into **Notion style**, **Anki cards**, or add a **step-by-step solution explanation**.


In [0]:
import pyspark.pandas as ps
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Define schema for Employee table
employee_schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("name", StringType(), False),
        StructField("departmentId", IntegerType(), False),
        StructField("salary", IntegerType(), False),
    ]
)

# Sample employee data
employee_data = [
    (1, "Joe", 1, 85000),
    (2, "Henry", 2, 80000),
    (3, "Sam", 2, 60000),
    (4, "Max", 1, 90000),
    (5, "Janet", 1, 69000),
    (6, "Randy", 1, 85000),
    (7, "Will", 1, 70000),
]

# Create Spark DataFrame for Employee
df_employee = spark.createDataFrame(employee_data, schema=employee_schema)
display(df_employee)

# Define schema for Department table
dept_schema = StructType(
    [StructField("id", IntegerType(), False), StructField("name", StringType(), False)]
)

# Sample department data
dept_data = [(1, "IT"), (2, "Sales")]

# Create Spark DataFrame for Department
df_dept = spark.createDataFrame(dept_data, schema=dept_schema)
display(df_dept)

# Convert Spark DataFrames to pandas-on-Spark DataFrames
df_employee_pandas = df_employee.pandas_api()
df_dept_pandas = df_dept.pandas_api()

# Register DataFrames as temporary views for SQL queries
df_employee.createOrReplaceTempView("Employee")
df_dept.createOrReplaceTempView("Department")

In [0]:
%sql
-- CTE to rank employees by salary within each department
with cte_emp_dept_salary as (
  select
    d.name as Department,
    e.name as Employee,
    e.salary as Salary,
    dense_rank() over (partition by d.name order by e.salary desc) as rnk -- Rank employees by salary (highest first) per department
  from
    Employee as e
      inner join Department as d
        on e.departmentId = d.id -- Join employees to their departments
)
-- Select top 3 highest paid employees per department
select
  Department,
  Employee,
  Salary
from
  cte_emp_dept_salary
where
  rnk <= 3 -- Only include top 3 salaries per department